# النتائج والتحليل (Result & Analysis) — التقييم الكمي لأداء DMorphNet

هذا الدفتر ينفّذ قسم **"Result and Analysis — Quantitative Performance Evaluation"**: التقييم التجريبي الكامل لنظام DMorphNet (سمات EfficientNet-B6 + مصنّف SVM) على مجموعة اختبار منفصلة (2,000 صورة: 1,000 حقيقية + 1,000 مدموجة، بهويات لم يرها النظام).

## المقاييس المستخدمة
- **الدقة (Accuracy)**، **الضبط (Precision)**، **الاستدعاء (Recall)**، **F1-Score**.
- **مصفوفة الالتباس (Confusion Matrix)** — كم صورة صُنّفت صحيحاً وكم خطأً (الشكل ٢).
- **منحنى ROC ومساحة AUC**.
- **مقارنة مع إصدارات EfficientNet الأخرى** (B0, B3, B6) لإثبات فاعلية المعمارية المقترحة.

## النتائج المرجعية المنشورة في الورقة (للمقارنة)

| الفئة الفعلية | تنبؤ: حقيقية | تنبؤ: مدموجة |
|---|---|---|
| صور حقيقية | 852 (TN) | 148 (FP) |
| صور مدموجة | 54 (FN) | 946 (TP) |

بدقة إجمالية **89.9%**. سنحسب النتائج نفسها على نظامنا ونقارنها بهذه القيم.


## ١) تجهيز البيئة


In [ ]:
!pip -q install scikit-learn joblib pandas

import os, glob, random, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# القيم المرجعية من الورقة (الجدول ١ / الشكل ٢)
PAPER = {"TN": 852, "FP": 148, "FN": 54, "TP": 946, "ACC": 0.899}
print("✅ البيئة جاهزة")


## ٢) تحميل السمات والمصنّف النهائي

نحمّل سمات مجموعات (التدريب/التحقق/الاختبار) ومصنّف SVM النهائي والـ Scaler من الدفاتر السابقة (محلياً أو من Google Drive). إن لم يوجد مصنّف محفوظ، نعيد تدريبه سريعاً من السمات.


In [ ]:
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

FEAT_DIR = "/content/DMorphNet_features"
DRIVE_DIR = "/content/drive/MyDrive/DMorphNet_models"
SPLITS = ["train", "val", "test"]


def find_file(name):
    for root in [FEAT_DIR, DRIVE_DIR]:
        p = os.path.join(root, name)
        if os.path.exists(p):
            return p
    return None


if find_file("effb6_ft_train.npz") is None and find_file("effb6_train.npz") is None:
    from google.colab import drive
    drive.mount('/content/drive')

PREFIX = "effb6_ft" if find_file("effb6_ft_train.npz") else "effb6"
F, y = {}, {}
for split in SPLITS:
    data = np.load(find_file(f"{PREFIX}_{split}.npz"))
    F[split] = data["X"].astype(np.float32)
    y[split] = np.where(data["y"] == 0, -1, +1)     # -1 حقيقية ، +1 مدموجة

svm_path, sc_path = find_file("svm_hybrid_final.joblib"), find_file("scaler_hybrid_final.joblib")
if svm_path and sc_path:
    svm_final, scaler = joblib.load(svm_path), joblib.load(sc_path)
    print("تم تحميل المصنّف النهائي المحفوظ")
else:
    print("لا يوجد مصنّف محفوظ — إعادة تدريب سريعة ...")
    scaler = StandardScaler().fit(F["train"])
    svm_final = LinearSVC(C=1.0, random_state=SEED)
    svm_final.fit(scaler.transform(F["train"]), y["train"])

Fs = {s: scaler.transform(F[s]) for s in SPLITS}
print("سمات الاختبار:", F["test"].shape,
      "— حقيقية:", int((y["test"] == -1).sum()),
      "مدموجة:", int((y["test"] == +1).sum()))


## ٣) مصفوفة الالتباس (الشكل ٢ والجدول ١)

نصنّف مجموعة الاختبار ونرسم مصفوفة الالتباس بنفس أسلوب الشكل ٢ في الورقة (تدرّج أزرق + شريط ألوان)، ونعرض الجدول ١ بقيمنا مقابل القيم المنشورة.


In [ ]:
from sklearn.metrics import confusion_matrix

pred_test = svm_final.predict(Fs["test"])
score_test = svm_final.decision_function(Fs["test"])

# الترتيب: الصف الأول Real ، الثاني Morph (مثل الورقة)
cm = confusion_matrix(y["test"], pred_test, labels=[-1, +1])
TN, FP = int(cm[0, 0]), int(cm[0, 1])
FN, TP = int(cm[1, 0]), int(cm[1, 1])

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Blues")
plt.colorbar(im, ax=ax)
ax.set_title("Confusion Matrix")
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(["Real", "Morph"]); ax.set_yticklabels(["Real", "Morph"])
ax.set_xlabel("Predicted label"); ax.set_ylabel("True label")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=14,
                color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.tight_layout()
plt.show()

# الجدول ١: نتائجنا مقابل القيم المنشورة في الورقة
table1 = pd.DataFrame({
    "الفئة الفعلية": ["صور حقيقية", "صور مدموجة"],
    "تنبؤ: حقيقية (نتيجتنا)": [TN, FN],
    "تنبؤ: مدموجة (نتيجتنا)": [FP, TP],
    "تنبؤ: حقيقية (الورقة)": [PAPER["TN"], PAPER["FN"]],
    "تنبؤ: مدموجة (الورقة)": [PAPER["FP"], PAPER["TP"]],
})
table1


## ٤) قراءة قيم مصفوفة الالتباس (TP / TN / FP / FN)

كما في الورقة:
- **TP (إيجابي صحيح)**: صور مدموجة صُنّفت مدموجة ✅
- **TN (سلبي صحيح)**: صور حقيقية صُنّفت حقيقية ✅
- **FP (إيجابي خاطئ)**: صور حقيقية صُنّفت مدموجة ❌ — مقبول نسبياً في التطبيقات الأمنية لأن المشتبه به يُراجع يدوياً.
- **FN (سلبي خاطئ)**: صور مدموجة صُنّفت حقيقية ❌ — **الأخطر** في الأنظمة البيومترية: مورف غير مكتشف قد يسمح بانتحال هوية.


In [ ]:
print("قيم مصفوفة الالتباس لنظامنا:")
print(f"  TP = {TP:4d} صورة مدموجة صُنّفت مدموجة (صحيح)     — الورقة: {PAPER['TP']}")
print(f"  TN = {TN:4d} صورة حقيقية صُنّفت حقيقية (صحيح)     — الورقة: {PAPER['TN']}")
print(f"  FP = {FP:4d} صورة حقيقية صُنّفت مدموجة (خطأ)       — الورقة: {PAPER['FP']}")
print(f"  FN = {FN:4d} صورة مدموجة صُنّفت حقيقية (خطأ خطِر)  — الورقة: {PAPER['FN']}")

accuracy = (TP + TN) / (TP + TN + FP + FN)
print(f"\nالدقة الإجمالية لنظامنا: {accuracy:.4f} ({accuracy*100:.1f}%)")
print(f"الدقة المنشورة في الورقة: {PAPER['ACC']*100:.1f}%")


## ٥) المقاييس الكاملة: الدقة والضبط والاستدعاء و F1

المقاييس محسوبة من مصفوفة الالتباس (الفئة الإيجابية = مدموجة):

$$\text{Accuracy}=\frac{TP+TN}{TP+TN+FP+FN}, \quad \text{Precision}=\frac{TP}{TP+FP}$$

$$\text{Recall}=\frac{TP}{TP+FN}, \quad F1 = 2\cdot\frac{\text{Precision}\cdot\text{Recall}}{\text{Precision}+\text{Recall}}$$


In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report)

metrics = {
    "Accuracy":  accuracy_score(y["test"], pred_test),
    "Precision": precision_score(y["test"], pred_test, pos_label=+1),
    "Recall":    recall_score(y["test"], pred_test, pos_label=+1),
    "F1-Score":  f1_score(y["test"], pred_test, pos_label=+1),
}
print("مقاييس الأداء على مجموعة الاختبار:")
for name, val in metrics.items():
    print(f"  {name:10s} = {val:.4f}")

print("\nالتقرير التفصيلي لكل فئة:")
print(classification_report(y["test"], pred_test,
                            target_names=["حقيقية (Real)", "مدموجة (Morph)"]))


## ٦) منحنى ROC ومساحة AUC

نرسم منحنى ROC من قيم دالة القرار $w^TF+b$ — كلما اقترب المنحنى من الزاوية العلوية اليسرى (AUC أقرب لـ 1) كان الفصل بين الحقيقي والمدموج أفضل عند كل عتبات القرار.


In [ ]:
from sklearn.metrics import roc_curve, auc

fpr, tpr, thresholds = roc_curve(y["test"], score_test, pos_label=+1)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, linewidth=2, label=f"DMorphNet (AUC = {roc_auc:.4f})")
plt.plot([0, 1], [0, 1], "--", color="gray", label="تخمين عشوائي (AUC = 0.5)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve — EfficientNet-B6 + SVM")
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"مساحة تحت المنحنى AUC = {roc_auc:.4f}")


## ٧) المقارنة مع إصدارات EfficientNet الأخرى

كما في الورقة، نختبر فاعلية النظام مقابل إصدارات أخرى من EfficientNet — نفس المسار تماماً (سمات مجمّدة ← Scaler ← SVM خطي) مع تغيير العمود الفقري فقط:

| النموذج | حجم الإدخال | ملاحظة |
|---|---|---|
| EfficientNet-B0 | 224×224 | الأصغر والأسرع |
| EfficientNet-B3 | 300×300 | وسط |
| EfficientNet-B6 | 528×528 | **المقترح** (سماتنا الجاهزة) |

للتسريع: التدريب على عيّنة 6,000 صورة من التدريب، والتقييم على **كامل** مجموعة الاختبار (2,000).

> ⚙️ تحتاج هذه الخلية GPU وصور `DMorphNet_processed` (سيفك ضغطها من Drive إن لم تكن موجودة).


In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0, EfficientNetB3
from tensorflow.keras.applications.efficientnet import preprocess_input

PROC_DIR = "/content/DMorphNet_processed"
if not os.path.isdir(os.path.join(PROC_DIR, "train")):
    from google.colab import drive
    drive.mount('/content/drive')
    shutil.unpack_archive("/content/drive/MyDrive/DMorphNet_processed.zip", PROC_DIR)

LABEL_MAP = {"real": 0, "morph": 1}
AUTOTUNE = tf.data.AUTOTUNE


def list_files(split):
    paths, labels = [], []
    for cls, lab in LABEL_MAP.items():
        fs = sorted(glob.glob(os.path.join(PROC_DIR, split, cls, "*.jpg")))
        paths += fs
        labels += [lab] * len(fs)
    return paths, np.array(labels)


tr_paths, tr_y01 = list_files("train")
te_paths, te_y01 = list_files("test")
rng = np.random.RandomState(SEED)
tr_idx = rng.permutation(len(tr_paths))[:6000]
sub_tr_paths = [tr_paths[i] for i in tr_idx]
y_tr = np.where(tr_y01[tr_idx] == 0, -1, +1)
y_te = np.where(te_y01 == 0, -1, +1)


def extract(model_cls, size, paths):
    base = model_cls(include_top=False, weights="imagenet",
                     pooling="avg", input_shape=(size, size, 3))

    def load(p):
        img = tf.io.decode_jpeg(tf.io.read_file(p), channels=3)
        img = tf.image.resize(tf.cast(img, tf.float32), [size, size])
        return preprocess_input(img)

    ds = (tf.data.Dataset.from_tensor_slices(list(paths))
          .map(load, num_parallel_calls=AUTOTUNE).batch(32).prefetch(AUTOTUNE))
    return base.predict(ds, verbose=1)


def evaluate_backbone(X_tr, X_te):
    sc = StandardScaler().fit(X_tr)
    clf = LinearSVC(C=1.0, random_state=SEED).fit(sc.transform(X_tr), y_tr)
    pred = clf.predict(sc.transform(X_te))
    return {
        "Accuracy":  accuracy_score(y_te, pred),
        "Precision": precision_score(y_te, pred, pos_label=+1),
        "Recall":    recall_score(y_te, pred, pos_label=+1),
        "F1-Score":  f1_score(y_te, pred, pos_label=+1),
    }


comparison = {}
print("--- EfficientNet-B0 (224) ---")
comparison["EfficientNet-B0"] = evaluate_backbone(
    extract(EfficientNetB0, 224, sub_tr_paths),
    extract(EfficientNetB0, 224, te_paths))

print("--- EfficientNet-B3 (300) ---")
comparison["EfficientNet-B3"] = evaluate_backbone(
    extract(EfficientNetB3, 300, sub_tr_paths),
    extract(EfficientNetB3, 300, te_paths))

print("--- EfficientNet-B6 (528) — سمات جاهزة ---")
comparison["EfficientNet-B6 (المقترح)"] = evaluate_backbone(
    F["train"][tr_idx], F["test"])

cmp_df = pd.DataFrame(comparison).T.round(4)
print("\nجدول المقارنة بين الإصدارات:")
cmp_df


## ٨) رسم بياني للمقارنة

مخطط أعمدة يقارن المقاييس الأربعة للإصدارات الثلاثة — لتأكيد تفوّق B6 المقترح بصرياً.


In [ ]:
ax = cmp_df.plot(kind="bar", figsize=(10, 5), rot=15, width=0.75)
ax.set_title("مقارنة إصدارات EfficientNet في كشف المورف (سمات + SVM)")
ax.set_ylabel("قيمة المقياس")
ax.set_ylim(0.5, 1.0)
ax.legend(loc="lower right")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

best = cmp_df["Accuracy"].idxmax()
print(f"🏆 الأفضل من حيث الدقة: {best} ({cmp_df.loc[best, 'Accuracy']:.4f})")


## ٩) تحليل الأخطاء بصرياً (FN و FP)

نعرض عيّنات من أخطاء النظام لفهم أين يفشل:
- **FN — الأخطر**: صور مدموجة اعتُبرت حقيقية (قد تسمح بانتحال هوية في نظام بيومتري حقيقي).
- **FP — مقبول**: صور حقيقية اعتُبرت مدموجة (تُحل بمراجعة يدوية للصور المشتبه بها).


In [ ]:
te_paths_arr = np.array(te_paths)
fn_idx = np.where((y["test"] == +1) & (pred_test == -1))[0]   # مورف فات النظام
fp_idx = np.where((y["test"] == -1) & (pred_test == +1))[0]   # حقيقي اشتُبه به

print(f"عدد FN (مورف صُنّف حقيقياً): {len(fn_idx)}")
print(f"عدد FP (حقيقي صُنّف مورفاً): {len(fp_idx)}")

import cv2
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for row, (idxs, title) in enumerate([(fn_idx, "FN: مدموجة صُنّفت حقيقية (الأخطر)"),
                                     (fp_idx, "FP: حقيقية صُنّفت مدموجة")]):
    show = idxs[:5] if len(idxs) >= 5 else idxs
    for col in range(5):
        ax = axes[row, col]
        if col < len(show):
            i = show[col]
            ax.imshow(cv2.cvtColor(cv2.imread(te_paths_arr[i]), cv2.COLOR_BGR2RGB))
            ax.set_title(f"wᵀF+b = {score_test[i]:+.2f}", fontsize=9)
        if col == 0:
            ax.set_ylabel(title, fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()


## ١٠) الخلاصة وحفظ النتائج

نلخّص كل النتائج في ملف CSV ونحفظه مع الرسوم في Google Drive.

**الاستنتاج (كما في الورقة):** نظام DMorphNet (EfficientNet-B6 + SVM) يميّز بين الوجوه الحقيقية والمدموجة بدقة عالية (~89.9% في الورقة). أخطاء FN القليلة مهمة جداً في الأنظمة البيومترية لأن أي مورف غير مكتشف قد يتيح انتحال هوية، بينما FP الأعلى قليلاً مقبول أمنياً لأن الصور المشتبه بها تُراجع يدوياً.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RESULTS_DIR = "/content/drive/MyDrive/DMorphNet_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

summary = pd.DataFrame([{
    "TP": TP, "TN": TN, "FP": FP, "FN": FN,
    "Accuracy": metrics["Accuracy"],
    "Precision": metrics["Precision"],
    "Recall": metrics["Recall"],
    "F1": metrics["F1-Score"],
    "AUC": roc_auc,
    "Paper_Accuracy": PAPER["ACC"],
}])
summary.to_csv(os.path.join(RESULTS_DIR, "test_results_summary.csv"), index=False)
cmp_df.to_csv(os.path.join(RESULTS_DIR, "efficientnet_versions_comparison.csv"))

print("تم حفظ النتائج في:", RESULTS_DIR)
print(os.listdir(RESULTS_DIR))
print("\n🎉 اكتمل التقييم الكمي لنظام DMorphNet")
summary.round(4)
